# Walkthrough ทีละ block
Notebook นี้เรียกฟังก์ชันของแต่ละ block ตามลำดับ และแสดงกราฟของ block นั้น ๆ (ใช้ค่าจาก `config/weather.yaml`)

In [ ]:
%matplotlib inline
import sys, numpy as np
sys.path.insert(0, '..')
from radar_sim import load_config, plots
from radar_sim.simulate import make_cells, simulate_if_path
from radar_sim.validation import simulate_dwells, if_vs_baseband
from radar_sim.processing import process
from radar_sim.scene import rain_profile, expected_measured_dbz
from radar_sim.interpret import fig_interpretation
cfg = load_config('../config/weather.yaml')
print(cfg.d.summary())

## Block 1 — waveform

In [ ]:
plots.fig_waveform(cfg)

## Block 2 — scene (ground truth)

In [ ]:
cells = make_cells(cfg)
plots.fig_scene(cfg, cells)

## Blocks 3–8 — channel, down-converter, noise, BPF, ADC, DDC (IF path ความละเอียดเต็ม 2 chirp)

In [ ]:
ifp = simulate_if_path(cfg, cells=cells)
plots.fig_channel(cfg, ifp['channel'])

In [ ]:
plots.fig_downconverter(cfg, ifp)

In [ ]:
plots.fig_noise(cfg, ifp['channel'])

In [ ]:
plots.fig_bpf(cfg, ifp)

In [ ]:
plots.fig_adc(cfg, ifp)

In [ ]:
plots.fig_ddc(cfg, ifp)

### ตรวจว่า complex-baseband model ให้ผลเท่ากับ IF path

In [ ]:
x_if, x_bb, nmse = if_vs_baseband(cfg, cells)
print(f'NMSE = {nmse:.1f} dB')
plots.fig_equivalence(cfg, x_if, x_bb, nmse)

## Blocks 9–17 — processing (ใช้ baseband model, หลาย dwell)

In [ ]:
frames, ch, _ = simulate_dwells(cfg, first_cells=cells)
res = process(frames, cfg)
truth = rain_profile(cfg, np.linspace(1, cfg.processing.r_max, 3000))
expected = expected_measured_dbz(cfg, res['ranges'], cfg.processing.range_window)

In [ ]:
plots.fig_dechirp(cfg, res)

In [ ]:
plots.fig_beat_lpf(cfg, res)

In [ ]:
plots.fig_range(cfg, res)

In [ ]:
plots.fig_clutter(cfg, res)

In [ ]:
plots.fig_doppler(cfg, res, truth)

In [ ]:
plots.fig_detection(cfg, res)

In [ ]:
plots.fig_reflectivity(cfg, res, truth, expected)

In [ ]:
plots.fig_attenuation(cfg, res, truth)

In [ ]:
plots.fig_rainrate(cfg, res, truth)

## ตีความ spectrogram

In [ ]:
fig_interpretation(cfg, res, truth)

## ลองเปลี่ยนพารามิเตอร์
ตัวอย่าง: ใส่ LNA และเพิ่ม idle time แล้วดู v_max, NF และ warning ที่เปลี่ยนไป (LNA เพิ่ม gain อีก 20 dB ทำให้ leakage ที่ ADC เกิน full scale จึงต้องลด `rx_gain_db` ตามที่ทำใน `config/weather_lna.yaml`)

In [ ]:
cfg2 = cfg.replace(frontend={'lna': {'enabled': True}}, waveform={'t_idle': 200e-6})
print(cfg2.d.summary())